# U9-02 · Caso práctico (2): preparación y modelado

*Especialista en Inteligencia Artificial (IFCD107) · Unidad 9*

Ejecuta cada celda con **Mayús + Intro** (en Colab, también con ▶). Con GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución*) las redes se entrenan mucho más deprisa.

> Datos: Moro, S., Rita, P. y Cortez, P. (2014). *Bank Marketing*. UCI, CC BY 4.0. https://doi.org/10.24432/C5K306 · Moro, S., Cortez, P. y Rita, P. (2014). *Decision Support Systems*, 62, 22–31. https://doi.org/10.1016/j.dss.2014.03.001

In [ ]:
%pip install -q "flaml[automl]"
# Descarga los datos y módulos auxiliares (solo la primera vez)
import os, urllib.request
for f in ["bank-additional-full.csv", "banco.py", "evaluar.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve("https://machinelearning-formacion.github.io/ifcd107/recursos/datos/u9/" + f, f)

## 9.3 · Preparación y métricas

`evaluar.py`

In [ ]:
"""Preparación y métricas comunes del caso práctico."""
import numpy as np
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from banco import num, cat

def prep_lineal():                        # escalar y one-hot
    return make_column_transformer(
        (StandardScaler(), num),
        (OneHotEncoder(handle_unknown="ignore"), cat))

def mejora(y, p, frac=0.1):               # lift del mejor 10 %
    n = int(frac * len(y))
    top = np.argsort(-np.asarray(p))[:n]
    return np.asarray(y)[top].mean() / np.mean(y)

def metricas(y, p):
    return (f"AUC {roc_auc_score(y, p):.3f}  AP"
            f" {average_precision_score(y, p):.3f}"
            f"  lift@10% {mejora(y, p):.2f}")

from sklearn.compose import make_column_transformer as _mct
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier

def potenciacion(**kw):                   # el modelo de referencia
    ordinal = _mct(("passthrough", num),
                   (OrdinalEncoder(handle_unknown="use_encoded_value",
                                   unknown_value=-1), cat))
    k = len(num)                          # las categóricas van detrás
    opciones = dict(categorical_features=list(range(k, k + len(cat))),
                    learning_rate=0.05, max_iter=300, random_state=0)
    opciones.update(kw)
    return make_pipeline(ordinal,
                         HistGradientBoostingClassifier(**opciones))

num_todas, cat_todas = list(num), list(cat)   # para quitar variables

`y1_referencias.py`

In [ ]:
import numpy as np
from banco import X_p, y_p
from evaluar import metricas

print("tasa de suscripción en la prueba:", round(y_p.mean(), 3))
print("acierto diciendo siempre 'no':", round(1 - y_p.mean(), 3))
azar = np.random.default_rng(0).random(len(y_p))
print("al azar:              ", metricas(y_p, azar))
regla = (X_p.poutcome == "success") + 0.5 * (X_p.contact ==
                                             "cellular")
print("regla de 2 variables: ", metricas(y_p, regla))

## 9.4 · Modelado

`y2_logistica.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from banco import X_e, X_p, y_e, y_p
from evaluar import prep_lineal, metricas

m = make_pipeline(prep_lineal(), LogisticRegression(max_iter=3000))
m.fit(X_e, y_e)
p = m.predict_proba(X_p)[:, 1]
print("regresión logística:", metricas(y_p, p))
nombres = m[0].get_feature_names_out()
coef = pd.Series(m[-1].coef_[0], nombres).sort_values()
print("razón de probabilidades (odds) de algunas variables:")
for v in ["onehotencoder__poutcome_success",
          "onehotencoder__contact_telephone",
          "onehotencoder__month_mar", "standardscaler__campaign"]:
    print(f"  {v.split('__')[1]:<17} {np.exp(coef[v]):5.2f}")

`y3_arbol.py`

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier, export_text
from banco import X_e, X_p, y_e, y_p, num, cat
from evaluar import metricas

prep = make_column_transformer(("passthrough", num),
                               (OrdinalEncoder(), cat))
for prof in [2, 4, 8, 16]:
    a = make_pipeline(prep, DecisionTreeClassifier(
        max_depth=prof, min_samples_leaf=50, random_state=0))
    a.fit(X_e, y_e)
    print(f"profundidad {prof:2d}:",
          metricas(y_p, a.predict_proba(X_p)[:, 1]))
a = make_pipeline(prep, DecisionTreeClassifier(
    max_depth=2, min_samples_leaf=50, random_state=0)).fit(X_e, y_e)
print(export_text(a[-1], feature_names=num + cat,
                  show_weights=True, decimals=2))

`y4_conjuntos.py`

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import (RandomForestClassifier,
                              HistGradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, StratifiedKFold
from banco import X_e, y_e, num, cat
from evaluar import prep_lineal

ordinal = make_column_transformer(
    ("passthrough", num),
    (OrdinalEncoder(handle_unknown="use_encoded_value",
                    unknown_value=-1), cat))
modelos = {
    "logística": make_pipeline(prep_lineal(),
                               LogisticRegression(max_iter=3000)),
    "bosque": make_pipeline(ordinal, RandomForestClassifier(
        500, min_samples_leaf=10, n_jobs=-1, random_state=0)),
    "potenciación": make_pipeline(ordinal,
        HistGradientBoostingClassifier(
            categorical_features=list(range(9, 19)),
            learning_rate=0.05, max_iter=300, random_state=0))}
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for n, m in modelos.items():
    r = cross_validate(m, X_e, y_e, cv=cv,
                       scoring=["roc_auc", "average_precision"])
    print(f"{n:<13} AUC {r['test_roc_auc'].mean():.3f}"
          f" ± {r['test_roc_auc'].std():.3f}"
          f"  AP {r['test_average_precision'].mean():.3f}"
          f"  ({r['fit_time'].mean():.1f} s)")

`y5_desequilibrio.py`

In [ ]:
from banco import X_e, X_p, y_e, y_p
from evaluar import potenciacion, metricas

for peso in [None, "balanced"]:
    m = potenciacion(class_weight=peso).fit(X_e, y_e)
    p = m.predict_proba(X_p)[:, 1]
    pred = m.predict(X_p)
    print(f"pesos {str(peso):<9}", metricas(y_p, p))
    print(f"   probabilidad media {p.mean():.3f}"
          f" (real {y_p.mean():.3f})"
          f" | marcados 'sí' {pred.mean():.1%}"
          f" | sensibilidad {pred[y_p == 1].mean():.3f}")

`y6_busqueda.py`

In [ ]:
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV
from banco import X_e, X_p, y_e, y_p
from evaluar import potenciacion, metricas

espacio = {
    "histgradientboostingclassifier__learning_rate":
        loguniform(0.01, 0.3),
    "histgradientboostingclassifier__max_leaf_nodes": randint(4, 64),
    "histgradientboostingclassifier__min_samples_leaf":
        randint(10, 300),
    "histgradientboostingclassifier__l2_regularization":
        loguniform(1e-3, 10)}
b = RandomizedSearchCV(potenciacion(), espacio, n_iter=25, cv=5,
                       scoring="average_precision", random_state=0,
                       n_jobs=-1).fit(X_e, y_e)
print(f"AP de validación: mejor {b.best_score_:.3f},"
      f" peor {b.cv_results_['mean_test_score'].min():.3f}")
for k, v in b.best_params_.items():
    print(f"  {k.split('__')[1]}: {round(v, 4)}")
print("prueba:", metricas(y_p, b.predict_proba(X_p)[:, 1]))

`y9_variables.py`

In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score, StratifiedKFold
from banco import X_e, y_e
from evaluar import potenciacion
import evaluar

def nuevas(X):                             # variables derivadas
    X = X.copy()
    X["contactado_antes"] = (X.pdays != 999).astype(int)
    X["pdays"] = X.pdays.where(X.pdays != 999, -1)
    X["insistencia"] = (X.campaign > 3).astype(int)
    return X
cv = StratifiedKFold(5, shuffle=True, random_state=0)
base = cross_val_score(potenciacion(), X_e, y_e, cv=cv,
                       scoring="average_precision").mean()
evaluar.num += ["contactado_antes", "insistencia"]
otra = cross_val_score(potenciacion(), nuevas(X_e), y_e, cv=cv,
                       scoring="average_precision").mean()
print(f"AP con las variables originales: {base:.4f}")
print(f"AP con tres cambios de variables: {otra:.4f}")

In [ ]:
import evaluar                 # restaura las 19 variables
evaluar.num[:] = evaluar.num_todas
evaluar.cat[:] = evaluar.cat_todas

`y10_sin_contexto.py`

In [ ]:
from banco import X_e, X_p, X_f, y_e, y_p, y_f
import evaluar
from evaluar import potenciacion, metricas

eco = ["emp.var.rate", "cons.price.idx", "cons.conf.idx",
       "euribor3m", "nr.employed"]
quitar = {"con contexto": [], "sin contexto": eco,
          "sin contexto ni mes": eco + ["month"]}
for nombre, q in quitar.items():
    evaluar.num[:] = [c for c in evaluar.num_todas if c not in q]
    evaluar.cat[:] = [c for c in evaluar.cat_todas if c not in q]
    m = potenciacion().fit(X_e, y_e)
    print(f"{nombre}:")
    for t, X, y in [("prueba", X_p, y_p), ("2010  ", X_f, y_f)]:
        print(f"   {t} {metricas(y, m.predict_proba(X)[:, 1])}")

In [ ]:
import evaluar                 # restaura las 19 variables
evaluar.num[:] = evaluar.num_todas
evaluar.cat[:] = evaluar.cat_todas

`y12_seleccion.py`

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from banco import X_e, y_e
import evaluar
from evaluar import potenciacion

cv = StratifiedKFold(5, shuffle=True, random_state=0)
def ap(quitar):
    evaluar.num[:] = [c for c in evaluar.num_todas if c not in quitar]
    evaluar.cat[:] = [c for c in evaluar.cat_todas if c not in quitar]
    return cross_val_score(potenciacion(), X_e, y_e, cv=cv,
                           scoring="average_precision").mean()
print(f"19 variables:             AP {ap([]):.4f}")
poco = ["marital", "education", "housing", "loan", "default"]
print(f"sin 5 variables del cliente: AP {ap(poco):.4f}")
solo = ["emp.var.rate", "euribor3m", "month", "pdays", "contact",
        "day_of_week", "poutcome", "campaign", "age", "job"]
resto = [c for c in evaluar.num_todas + evaluar.cat_todas
         if c not in solo]
print(f"solo 10 variables:        AP {ap(resto):.4f}")

In [ ]:
import evaluar                 # restaura las 19 variables
evaluar.num[:] = evaluar.num_todas
evaluar.cat[:] = evaluar.cat_todas

`y13_aprendizaje.py`

In [ ]:
import numpy as np
from sklearn.model_selection import learning_curve, StratifiedKFold
from banco import X_e, y_e
from evaluar import potenciacion

tam, ent, val = learning_curve(
    potenciacion(), X_e, y_e, train_sizes=[0.05, 0.1, 0.25, 0.5, 1.0],
    cv=StratifiedKFold(5, shuffle=True, random_state=0),
    scoring="average_precision")
for n, e, v in zip(tam, ent.mean(1), val.mean(1)):
    print(f"{n:6d} clientes: AP entrenamiento {e:.3f},"
          f" validación {v:.3f}")

`y11_temporal.py`

In [ ]:
from banco import d, X, y
from evaluar import potenciacion, metricas
from sklearn.model_selection import train_test_split

a8, a9 = (d.año == 2008).to_numpy(), (d.año == 2009).to_numpy()
m = potenciacion().fit(X[a8], y[a8])
print("entrenado con 2008, probado con 2009:")
print("  ", metricas(y[a9], m.predict_proba(X[a9])[:, 1]))
h = a8 | a9
Xe, Xp, ye, yp = train_test_split(X[h], y[h], test_size=0.3,
                                  stratify=y[h], random_state=0)
m = potenciacion().fit(Xe, ye)
print("2008 y 2009 mezclados al azar:")
print("  ", metricas(yp, m.predict_proba(Xp)[:, 1]))

`y7_red.py`

In [ ]:
import numpy as np
import keras
from banco import X_e, X_p, y_e, y_p
from evaluar import prep_lineal, metricas

prep = prep_lineal().fit(X_e)
Ze = np.asarray(prep.transform(X_e), "float32")
Zp = np.asarray(prep.transform(X_p), "float32")
keras.utils.set_random_seed(0)
m = keras.Sequential([keras.Input((Ze.shape[1],)),
                      keras.layers.Dense(64, activation="relu"),
                      keras.layers.Dropout(0.3),
                      keras.layers.Dense(32, activation="relu"),
                      keras.layers.Dense(1, activation="sigmoid")])
m.compile(keras.optimizers.Adam(1e-3), "binary_crossentropy",
          metrics=[keras.metrics.AUC(curve="PR", name="ap")])
parar = keras.callbacks.EarlyStopping(
    monitor="val_ap", mode="max", patience=8,
    restore_best_weights=True)
h = m.fit(Ze, y_e.to_numpy(), epochs=50, batch_size=256, verbose=0,
          validation_split=0.2, callbacks=[parar])
print("variables tras la preparación:", Ze.shape[1],
      "| épocas:", len(h.history["loss"]))
p = m.predict(Zp, verbose=0).ravel()
print("red neuronal:", metricas(y_p, p))

`y8_automl.py`

In [ ]:
from flaml import AutoML
from banco import X_e, X_p, y_e, y_p
from evaluar import metricas

auto = AutoML()
auto.fit(X_e, y_e, task="classification", metric="ap",
         time_budget=90, seed=0, verbose=0,
         estimator_list=["lgbm", "rf", "extra_tree", "xgboost",
                         "lrl1"])
print("mejor algoritmo:", auto.best_estimator,
      "| AP de validación:", round(1 - auto.best_loss, 3))
print("AutoML:", metricas(y_p, auto.predict_proba(X_p)[:, 1]))

**Ejercicio.** Entrena la potenciación solo con las diez variables de `y12_seleccion.py` y evalúala en 2010. ¿Es más robusta?